# Gold - CFPB API Incremental Load

## Load New Date Dimension

In [17]:
spark.sql("""
    MERGE INTO gold.dim_date AS target
    USING (
        SELECT DISTINCT
            date_received_clean AS date,
            YEAR(date_received_clean) AS year,
            QUARTER(date_received_clean) AS quarter,
            MONTH(date_received_clean) AS month_number,
            DATE_FORMAT(date_received_clean, 'MMMM') AS month_name,
            DATE_FORMAT(date_received_clean, 'yyyy-MM') AS year_month,
            DAYOFWEEK(date_received_clean) AS day_of_week_number,
            DATE_FORMAT(date_received_clean, 'EEEE') AS day_of_week_name,
            DAYOFWEEK(date_received_clean) IN (1, 7) AS is_weekend
        FROM silver.complaints
        WHERE source_file = 'cfpb_api'
    ) AS source
        ON target.date = source.date

    WHEN NOT MATCHED THEN INSERT *
""")

StatementMeta(, 7152585b-c2c9-4644-b08d-a6b86137dc04, 19, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Load New Location Dimension

In [18]:
spark.sql("""
    MERGE INTO gold.dim_location AS target
    USING (
        SELECT DISTINCT
            CONCAT(
                COALESCE(state_clean, 'Unknown'),
                '|',
                COALESCE(zip_code_clean, 'No ZIP'),
                '|',
                zip_code_status
            ) AS location_key,
            COALESCE(state_clean, 'Unknown') AS state,
            COALESCE(zip_code_clean, 'No ZIP') AS zip_code,
            zip_code_status
        FROM silver.complaints
        WHERE source_file = 'cfpb_api'
    ) AS source
        ON target.location_key = source.location_key

    WHEN NOT MATCHED THEN INSERT *
""")

StatementMeta(, 7152585b-c2c9-4644-b08d-a6b86137dc04, 20, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Load New Product Issue Dimension

In [20]:
spark.sql("""
    MERGE INTO gold.dim_product_issue AS target
    USING (
        SELECT DISTINCT
            CONCAT(
                COALESCE(product, 'Unknown'),
                '|',
                COALESCE(sub_product, 'No Sub-product'),
                '|',
                COALESCE(issue, 'Unknown'),
                '|',
                COALESCE(sub_issue, 'No Sub-issue')
            ) AS product_issue_key,
            COALESCE(product, 'Unknown') AS product,
            COALESCE(sub_product, 'No Sub-product') AS sub_product,
            COALESCE(issue, 'Unknown') AS issue,
            COALESCE(sub_issue, 'No Sub-issue') AS sub_issue
        FROM silver.complaints
        WHERE source_file = 'cfpb_api'
    ) AS source
        ON target.product_issue_key = source.product_issue_key

    WHEN NOT MATCHED THEN INSERT *
""")

StatementMeta(, 7152585b-c2c9-4644-b08d-a6b86137dc04, 22, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Load New Complaint Fact

In [22]:
spark.sql("""
    MERGE INTO gold.fact_complaints AS target
    USING (
        SELECT
            complaint_id,
            date_received_clean AS date_received,
            date_sent_to_company_clean AS date_sent_to_company,
            CONCAT(
                COALESCE(state_clean, 'Unknown'),
                '|',
                COALESCE(zip_code_clean, 'No ZIP'),
                '|',
                zip_code_status
            ) AS location_key,
            CONCAT(
                COALESCE(product, 'Unknown'),
                '|',
                COALESCE(sub_product, 'No Sub-product'),
                '|',
                COALESCE(issue, 'Unknown'),
                '|',
                COALESCE(sub_issue, 'No Sub-issue')
            ) AS product_issue_key,
            company,
            submitted_via,
            company_response_to_consumer,
            timely_response,
            DATEDIFF(date_sent_to_company_clean, date_received_clean) AS response_days
        FROM silver.complaints
        WHERE source_file = 'cfpb_api'
    ) AS source
        ON target.complaint_id = source.complaint_id

    WHEN NOT MATCHED THEN INSERT *
""")

StatementMeta(, 7152585b-c2c9-4644-b08d-a6b86137dc04, 24, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]